## 5-1 Split data into training and trade dataset
- Training data split: 2009-01-01 to 2018-12-31
- Trade data split: 2019-01-01 to 2020-09-30

In [ ]:
train = data_split(processed_full, '2009-01-01','2019-01-01')
trade = data_split(processed_full, '2019-01-01','2021-01-01')
# Check the length of the two datasets
print(len(train))
print(len(trade))

109530
21930


In [ ]:
train.head()

,date,tic,open,high,low,close,volume,OPM,NPM,ROA,ROE,cur_ratio,quick_ratio,cash_ratio,inv_turnover,acc_rec_turnover,acc_pay_turnover,debt_ratio,debt_to_equity,PE,PB,Div_yield
0,2009-01-02,AAPL,3.067143,3.251429,3.041429,2.782837,746015200.0,0.217886,0.163846,0.103222,0.183579,2.461857,2.039779,1.818995,54.403846,8.972003,4.269115,0.437727,0.778495,0.639733,0.102096,0.000000
0,2009-01-02,AXP,18.570000,19.520000,18.400000,15.657365,10955700.0,0.093973,0.072040,0.014094,0.108238,0.000000,0.000000,0.000000,0.000000,0.351354,0.653355,0.869784,6.679531,50.507629,1.450032,0.011496
0,2009-01-02,BA,42.799999,45.560001,42.779999,33.941101,7010200.0,0.047307,0.032525,0.026400,-2.870334,0.927883,0.368463,0.148507,2.329670,6.815203,2.076967,1.009198,-109.722986,39.012760,-35.751054,0.012374
0,2009-01-02,CAT,44.910000,46.980000,44.709999,32.655109,7117200.0,0.124545,0.066662,0.040891,0.415878,1.343293,0.890488,0.163158,3.540791,2.460351,8.472455,0.893715,9.089489,-171.868997,3.151894,0.012862
0,2009-01-02,CSCO,16.410000,17.000000,16.250000,12.505757,40980600.0,0.234698,0.196418,0.097593,0.162793,2.792929,2.498162,2.170759,9.054201,6.844634,16.036800,0.400215,0.667591,19.850408,1.986886,0.000000


In [ ]:
trade.head()

,date,tic,open,high,low,close,volume,OPM,NPM,ROA,ROE,cur_ratio,quick_ratio,cash_ratio,inv_turnover,acc_rec_turnover,acc_pay_turnover,debt_ratio,debt_to_equity,PE,PB,Div_yield
0,2019-01-01,AAPL,38.722500,39.712502,38.557499,38.382229,148158800.0,0.258891,0.227773,0.133360,0.430843,1.315382,1.134347,0.854114,23.571867,7.620024,3.781658,0.690466,2.230663,5.728691,1.670488,0.019019
0,2019-01-01,AXP,93.910004,96.269997,93.769997,91.803406,4175400.0,0.203479,0.160494,0.026811,0.237960,0.000000,0.000000,0.000000,0.000000,0.231669,0.279424,0.887329,7.875371,50.720114,3.458432,0.004248
0,2019-01-01,BA,316.190002,323.950012,313.709991,314.645142,3292200.0,0.116496,0.102682,0.066409,34.409483,1.070490,0.262465,0.092436,0.933164,5.468453,4.151637,0.998070,517.142241,83.019826,1418.196271,0.006531
0,2019-01-01,CAT,124.029999,127.879997,123.000000,117.506577,4783200.0,0.186871,0.107064,0.056932,0.289572,1.428582,0.919490,0.266175,2.135008,2.339630,3.660183,0.803394,4.086316,35.716285,4.351801,0.007319
0,2019-01-01,CSCO,42.279999,43.200001,42.209999,39.496738,23833500.0,0.263373,0.261680,0.098017,0.246218,1.801859,1.677431,1.370671,7.722516,4.244056,7.937160,0.601911,1.512001,28.011871,4.283841,0.008355


## 5-2 Set up the training environment

In [ ]:
ratio_list = ['OPM', 'NPM','ROA', 'ROE', 'cur_ratio', 'quick_ratio', 'cash_ratio', 'inv_turnover','acc_rec_turnover', 'acc_pay_turnover', 'debt_ratio', 'debt_to_equity',
       'PE', 'PB', 'Div_yield']

stock_dimension = len(train.tic.unique())
state_space = 1 + 2*stock_dimension + len(ratio_list)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 30, State Space: 511


In [ ]:
# Parameters for the environment
env_kwargs = {
    "hmax": 100, 
    "initial_amount": 1000000, 
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space, 
    "stock_dim": stock_dimension, 
    "tech_indicator_list": ratio_list, 
    "action_space": stock_dimension, 
    "reward_scaling": 1e-4
    
}

#Establish the training environment using StockTradingEnv() class
e_train_gym = StockTradingEnv(df = train, **env_kwargs)

## Environment for Training



In [ ]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


<a id='5'></a>
# Part 6: Implement DRL Algorithms


In [ ]:
# Set up the agent using DRLAgent() class using the environment created in the previous part
agent = DRLAgent(env = env_train)

### Model Training: 5 models, A2C DDPG, PPO, TD3, SAC


### Model 1: A2C


In [ ]:
agent = DRLAgent(env = env_train)
model_a2c = agent.get_model("a2c")

{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device


In [ ]:
trained_a2c = agent.train_model(model=model_a2c, 
                             tb_log_name='a2c',
                             total_timesteps=100000)

Logging to tensorboard_log/a2c/a2c_1
------------------------------------
| time/                 |          |
|    fps                | 60       |
|    iterations         | 100      |
|    time_elapsed       | 8        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -42.9    |
|    explained_variance | 0.157    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 165      |
|    std                | 1.01     |
|    value_loss         | 18.3     |
------------------------------------
------------------------------------
| time/                 |          |
|    fps                | 68       |
|    iterations         | 200      |
|    time_elapsed       | 14       |
|    total_timesteps    | 1000     |
| train/                |          |
|    entropy_loss       | -43      |
|    explained_variance | 0.0513   |
|    learning_rate      | 0.0007   |
|    n_updates          | 199      |
|

### Model 2: DDPG

In [ ]:
agent = DRLAgent(env = env_train)
model_ddpg = agent.get_model("ddpg")

{'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}
Using cuda device


In [ ]:
trained_ddpg = agent.train_model(model=model_ddpg, 
                             tb_log_name='ddpg',
                             total_timesteps=50000)

Logging to tensorboard_log/ddpg/ddpg_3
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 66       |
|    time_elapsed    | 218      |
|    total timesteps | 14604    |
| train/             |          |
|    actor_loss      | -40.1    |
|    critic_loss     | 213      |
|    learning_rate   | 0.001    |
|    n_updates       | 10953    |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 62       |
|    time_elapsed    | 463      |
|    total timesteps | 29208    |
| train/             |          |
|    actor_loss      | -14.9    |
|    critic_loss     | 2.92     |
|    learning_rate   | 0.001    |
|    n_updates       | 25557    |
---------------------------------
day: 3650, episode: 40
begin_total_asset: 1000000.00
end_total_asset: 3024712.17
total_reward: 2024712.17
total_cost: 999.00
total_trades: 72983


### Model 3: PPO

In [ ]:
agent = DRLAgent(env = env_train)
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.01,
    "learning_rate": 0.00025,
    "batch_size": 128,
}
model_ppo = agent.get_model("ppo",model_kwargs = PPO_PARAMS)

{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 128}
Using cpu device


In [ ]:
trained_ppo = agent.train_model(model=model_ppo, 
                             tb_log_name='ppo',
                             total_timesteps=50000)

Logging to tensorboard_log/ppo/ppo_1
-----------------------------
| time/              |      |
|    fps             | 76   |
|    iterations      | 1    |
|    time_elapsed    | 26   |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 75          |
|    iterations           | 2           |
|    time_elapsed         | 54          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.018077655 |
|    clip_fraction        | 0.231       |
|    clip_range           | 0.2         |
|    entropy_loss         | -42.6       |
|    explained_variance   | -0.0123     |
|    learning_rate        | 0.00025     |
|    loss                 | 3.1         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0292     |
|    std                  | 1           |
|    value_loss           | 7.21       

### Model 4: TD3

In [ ]:
agent = DRLAgent(env = env_train)
TD3_PARAMS = {"batch_size": 100, 
              "buffer_size": 1000000, 
              "learning_rate": 0.001}

model_td3 = agent.get_model("td3",model_kwargs = TD3_PARAMS)

{'batch_size': 100, 'buffer_size': 1000000, 'learning_rate': 0.001}
Using cpu device


In [ ]:
trained_td3 = agent.train_model(model=model_td3, 
                             tb_log_name='td3',
                             total_timesteps=30000)

Logging to tensorboard_log/td3/td3_1
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 26       |
|    time_elapsed    | 559      |
|    total timesteps | 14604    |
| train/             |          |
|    actor_loss      | 4.64     |
|    critic_loss     | 720      |
|    learning_rate   | 0.001    |
|    n_updates       | 10953    |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 22       |
|    time_elapsed    | 1289     |
|    total timesteps | 29208    |
| train/             |          |
|    actor_loss      | 10.6     |
|    critic_loss     | 14.2     |
|    learning_rate   | 0.001    |
|    n_updates       | 25557    |
---------------------------------
day: 3650, episode: 40
begin_total_asset: 1000000.00
end_total_asset: 3319811.55
total_reward: 2319811.55
total_cost: 998.99
total_trades: 51100
Sh

### Model 5: SAC

In [ ]:
agent = DRLAgent(env = env_train)
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 1000000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

model_sac = agent.get_model("sac",model_kwargs = SAC_PARAMS)

{'batch_size': 128, 'buffer_size': 1000000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device


In [ ]:
trained_sac = agent.train_model(model=model_sac, 
                             tb_log_name='sac',
                             total_timesteps=80000)

Logging to tensorboard_log/sac/sac_2
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 21       |
|    time_elapsed    | 685      |
|    total timesteps | 14604    |
| train/             |          |
|    actor_loss      | 172      |
|    critic_loss     | 28.6     |
|    ent_coef        | 0.0742   |
|    ent_coef_loss   | -126     |
|    learning_rate   | 0.0001   |
|    n_updates       | 14503    |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 20       |
|    time_elapsed    | 1401     |
|    total timesteps | 29208    |
| train/             |          |
|    actor_loss      | 9.68     |
|    critic_loss     | 9.81     |
|    ent_coef        | 0.0174   |
|    ent_coef_loss   | -173     |
|    learning_rate   | 0.0001   |
|    n_updates       | 29107    |
---------------------------------
day: 3650, 

### Trade



In [ ]:
trade = data_split(processed_full, '2019-01-01','2021-01-01')
e_trade_gym = StockTradingEnv(df = trade, **env_kwargs)
# env_trade, obs_trade = e_trade_gym.get_sb_env()

In [ ]:
trade.head()

,date,tic,open,high,low,close,volume,OPM,NPM,ROA,ROE,cur_ratio,quick_ratio,cash_ratio,inv_turnover,acc_rec_turnover,acc_pay_turnover,debt_ratio,debt_to_equity,PE,PB,Div_yield
0,2019-01-01,AAPL,38.722500,39.712502,38.557499,38.382229,148158800.0,0.258891,0.227773,0.133360,0.430843,1.315382,1.134347,0.854114,23.571867,7.620024,3.781658,0.690466,2.230663,5.728691,1.670488,0.019019
0,2019-01-01,AXP,93.910004,96.269997,93.769997,91.803406,4175400.0,0.203479,0.160494,0.026811,0.237960,0.000000,0.000000,0.000000,0.000000,0.231669,0.279424,0.887329,7.875371,50.720114,3.458432,0.004248
0,2019-01-01,BA,316.190002,323.950012,313.709991,314.645142,3292200.0,0.116496,0.102682,0.066409,34.409483,1.070490,0.262465,0.092436,0.933164,5.468453,4.151637,0.998070,517.142241,83.019826,1418.196271,0.006531
0,2019-01-01,CAT,124.029999,127.879997,123.000000,117.506577,4783200.0,0.186871,0.107064,0.056932,0.289572,1.428582,0.919490,0.266175,2.135008,2.339630,3.660183,0.803394,4.086316,35.716285,4.351801,0.007319
0,2019-01-01,CSCO,42.279999,43.200001,42.209999,39.496738,23833500.0,0.263373,0.261680,0.098017,0.246218,1.801859,1.677431,1.370671,7.722516,4.244056,7.937160,0.601911,1.512001,28.011871,4.283841,0.008355


In [ ]:
df_account_value, df_actions = DRLAgent.DRL_prediction(
    model=trained_ddpg, 
    environment = e_trade_gym)

hit end!


In [ ]:
df_account_value.shape

(731, 2)

In [ ]:
df_account_value.tail()

,date,account_value
726,2020-12-27,1.338573e+06
727,2020-12-28,1.338573e+06
728,2020-12-29,1.334876e+06
729,2020-12-30,1.340975e+06
730,2020-12-31,1.350643e+06


In [ ]:
df_actions.head()

,AAPL,AXP,BA,CAT,CSCO,CVX,DD,DIS,GS,HD,IBM,INTC,JNJ,JPM,KO,MCD,MMM,MRK,MSFT,NKE,PFE,PG,RTX,TRV,UNH,V,VZ,WBA,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019-01-01,0,0,0,100,100,0,100,0,100,100,100,100,100,0,100,0,100,0,0,100,100,100,100,100,0,100,100,100,100,0
2019-01-02,0,0,0,100,100,0,100,0,100,100,100,100,100,0,100,0,100,0,0,100,100,100,100,100,0,100,100,100,100,0
2019-01-03,0,0,0,100,100,0,100,0,100,100,100,100,100,0,100,0,100,0,0,100,100,100,100,100,0,100,100,100,100,0
2019-01-04,0,0,0,100,100,0,100,0,100,100,100,100,100,0,100,0,100,0,0,100,100,100,100,100,0,100,100,100,100,0
2019-01-05,0,0,0,100,100,0,100,0,100,100,100,100,100,0,100,0,100,0,0,100,100,100,100,100,0,100,100,100,100,0


<a id='6'></a>
# Part 7: Backtest Our Strategy


<a id='6.1'></a>
## 7.1 BackTestStats
pass in df_account_value, this information is stored in env class


In [ ]:
print("==============Get Backtest Results===========")
now = datetime.datetime.now().strftime('%Y%m%d-%Hh%M')

perf_stats_all = backtest_stats(account_value=df_account_value)
perf_stats_all = pd.DataFrame(perf_stats_all)
perf_stats_all.to_csv("./"+config.RESULTS_DIR+"/perf_stats_all_"+now+'.csv')

==============Get Backtest Results===========
Annual return          0.109179
Cumulative returns     0.350643
Annual volatility      0.210154
Sharpe ratio           0.599338
Calmar ratio           0.339833
Stability              0.296554
Max drawdown          -0.321272
Omega ratio            1.164859
Sortino ratio          0.841479
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.899170
Daily value at risk   -0.025977
dtype: float64


In [ ]:
#baseline stats
print("==============Get Baseline Stats===========")
baseline_df = get_baseline(
        ticker="^DJI", 
        start = '2019-01-01',
        end = '2021-01-01')

stats = backtest_stats(baseline_df, value_col_name = 'close')


==============Get Baseline Stats===========
[*********************100%***********************]  1 of 1 completed
Shape of DataFrame:  (505, 8)
Annual return          0.144674
Cumulative returns     0.310981
Annual volatility      0.274619
Sharpe ratio           0.631418
Calmar ratio           0.390102
Stability              0.116677
Max drawdown          -0.370862
Omega ratio            1.149365
Sortino ratio          0.870084
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.860710
Daily value at risk   -0.033911
dtype: float64


<a id='6.2'></a>
## 7.2 BackTestPlot